In [68]:
# 1. Import the existing src APIs and discover all result models.
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from evaluation import evaluate
from datahandler import DataLoader
from gnn import GCNNodeClassifier

RESULTS_DIR = SRC_DIR / "results"
SIMULATION_PATH = PROJECT_ROOT / "data" / "simulation_scores.csv"
result_paths = sorted(
    path for path in RESULTS_DIR.iterdir()
    if path.is_dir() and (path / "best_seed_predictions.csv").exists()
)

print(f"Discovered {len(result_paths)} result models:")
for path in result_paths:
    print(f"  {path.name}")

Discovered 8 result models:
  baseline_multimodal
  baseline_only_a
  baseline_only_b
  gcn_multimodal_identity
  gcn_multimodal_real
  gcn_multimodal_shuffled
  gcn_only_a
  gcn_only_b


In [69]:
# 2. Load every model's saved predictions and merge simulation scores once.
simulation_scores = pd.read_csv(SIMULATION_PATH)
model_predictions = {}

for result_path in result_paths:
    predictions = pd.read_csv(result_path / "best_seed_predictions.csv")
    merged = predictions.merge(
        simulation_scores,
        on=["subject_id", "node_id"],
        how="inner",
        validate="one_to_one",
    )
    model_predictions[result_path.name] = merged

print("Loaded prediction tables:")
for model_name, predictions in model_predictions.items():
    test_rows = predictions[predictions["split"] == "test"]
    print(f"  {model_name}: {len(predictions)} rows, {len(test_rows)} test rows")

Loaded prediction tables:
  baseline_multimodal: 9520 rows, 1904 test rows
  baseline_only_a: 9520 rows, 1904 test rows
  baseline_only_b: 9520 rows, 1904 test rows
  gcn_multimodal_identity: 9520 rows, 1904 test rows
  gcn_multimodal_real: 9520 rows, 1904 test rows
  gcn_multimodal_shuffled: 9520 rows, 1904 test rows
  gcn_only_a: 9520 rows, 1904 test rows
  gcn_only_b: 9520 rows, 1904 test rows


In [58]:
# Standalone simulation-score evaluation for every model and every split.
simulation_evaluation_frames = {}
simulation_evaluation_results = []

for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_predictions = predictions[predictions["split"] == split_name].copy()

        # Create a new dataframe using sim_score as predict_score.
        simulation_df = split_predictions[
            ["subject_id", "node_id", "real_label", "sim_score"]
        ].rename(columns={"sim_score": "predict_score"})
        simulation_evaluation_frames[(model_name, split_name)] = simulation_df

        result = evaluate(simulation_df.copy())
        simulation_evaluation_results.append({
            "model": model_name,
            "split": split_name,
            **result,
        })

simulation_metrics_table = (
    pd.DataFrame(simulation_evaluation_results)
    .set_index(["model", "split"])
    .sort_index()
)

print("Standalone simulation evaluation for all models and splits:")
print(simulation_metrics_table.round(4))
print("\\nRows evaluated by model and split:")
print({key: len(frame) for key, frame in simulation_evaluation_frames.items()})

Standalone simulation evaluation for all models and splits:
                                auprc   auroc  top_k_dice  prevalence
model                   split                                        
baseline_multimodal     test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val    0.4371  0.7722      0.5548      0.0714
baseline_only_a         test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val    0.4371  0.7722      0.5548      0.0714
baseline_only_b         test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val    0.4371  0.7722      0.5548      0.0714
gcn_multimodal_identity test   0.4048  0.7726      0.4563      0.0720
                        train  0.4147  0.7593      0.4970      0.0734
                        val   

In [63]:
# Correlation between model predict_score and sim_score for every split.
from scipy.stats import pearsonr, spearmanr

correlation_rows = []
for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_predictions = predictions[predictions["split"] == split_name]
        model_scores = split_predictions["predict_score"].to_numpy()
        simulation_scores_for_split = split_predictions["sim_score"].to_numpy()

        correlation_rows.append({
            "model": model_name,
            "split": split_name,
            "rows": len(split_predictions),
            "pearson_correlation": pearsonr(
                model_scores, simulation_scores_for_split
            ).statistic,
            "spearman_correlation": spearmanr(
                model_scores, simulation_scores_for_split
            ).statistic,
        })

correlation_table = (
    pd.DataFrame(correlation_rows)
    .set_index(["model", "split"])
    .sort_index()
)

print("Correlation between model predict_score and sim_score:")
print(correlation_table.round(4))

Correlation between model predict_score and sim_score:
                               rows  pearson_correlation  spearman_correlation
model                   split                                                 
baseline_multimodal     test   1904               0.2242                0.1531
                        train  5712               0.3121                0.1721
                        val    1904               0.2580                0.1708
baseline_only_a         test   1904               0.1799                0.1323
                        train  5712               0.2205                0.1406
                        val    1904               0.1457                0.1183
baseline_only_b         test   1904               0.1362                0.1488
                        train  5712               0.2618                0.1665
                        val    1904               0.1853                0.1692
gcn_multimodal_identity test   1904               0.2754                0.14

In [67]:
# Where model_score and sim_score disagree the most.
disagreement_rows = []

for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_df = predictions[predictions["split"] == split_name].copy()
        split_df["model_score"] = split_df["predict_score"]
        split_df["absolute_difference"] = (
            split_df["model_score"] - split_df["sim_score"]
        ).abs()
        split_df["model"] = model_name
        split_df["split_name"] = split_name
        disagreement_rows.append(split_df[
            [
                "model",
                "split_name",
                "subject_id",
                "node_id",
                "model_score",
                "sim_score",
                "absolute_difference",
            ]
        ])

disagreement_table = pd.concat(disagreement_rows, ignore_index=True)
most_discordant = disagreement_table.sort_values(
    "absolute_difference", ascending=False
).head(20)

print("Top 20 disagreements between model_score and sim_score:")
print(most_discordant.to_string(index=False, float_format=lambda value: f"{value:.4f}"))

Top 20 disagreements between model_score and sim_score:
                  model split_name subject_id  node_id  model_score  sim_score  absolute_difference
        baseline_only_b      train    sub-129       34       0.0000     1.0000               1.0000
        baseline_only_b       test    sub-081       51       0.0001     1.0000               0.9999
        baseline_only_b        val    sub-033       66       0.0003     1.0000               0.9997
    baseline_multimodal      train    sub-129       34       0.0003     1.0000               0.9997
        baseline_only_a       test    sub-027       32       0.0004     1.0000               0.9996
        baseline_only_b      train    sub-090       19       0.0007     1.0000               0.9993
        baseline_only_b        val    sub-023       64       0.0008     1.0000               0.9992
    baseline_multimodal       test    sub-027       32       0.0009     1.0000               0.9991
    baseline_multimodal        val    sub-02

In [71]:
# Evaluate the naive average of model_score and sim_score.
average_evaluation_results = []
average_evaluation_frames = {}

for model_name, predictions in model_predictions.items():
    for split_name in ["train", "val", "test"]:
        split_predictions = predictions[predictions["split"] == split_name].copy()

        average_df = split_predictions[
            ["subject_id", "node_id", "real_label", "predict_score", "sim_score"]
        ].rename(columns={"predict_score": "model_score"})
        average_df["predict_score"] = (
            average_df["model_score"] + average_df["sim_score"]
        ) / 2
        average_evaluation_frames[(model_name, split_name)] = average_df

        result = evaluate(average_df.copy())
        average_evaluation_results.append({
            "model": model_name,
            "split": split_name,
            **result,
        })

average_metrics_table = (
    pd.DataFrame(average_evaluation_results)
    .set_index(["model", "split"])
    .sort_index()
)

print("Naive average evaluation: (model_score + sim_score) / 2")
print(average_metrics_table.round(4))
print("\\nRows evaluated:")
print({key: len(frame) for key, frame in average_evaluation_frames.items()})

Naive average evaluation: (model_score + sim_score) / 2
                                auprc   auroc  top_k_dice  prevalence
model                   split                                        
baseline_multimodal     test   0.6041  0.8763      0.5548      0.0720
                        train  0.6778  0.9197      0.6202      0.0734
                        val    0.6071  0.8564      0.6248      0.0714
baseline_only_a         test   0.5682  0.8631      0.5106      0.0720
                        train  0.6184  0.9020      0.5880      0.0734
                        val    0.5806  0.8540      0.6021      0.0714
baseline_only_b         test   0.5242  0.8118      0.5343      0.0720
                        train  0.6477  0.9129      0.6127      0.0734
                        val    0.5434  0.8164      0.5736      0.0714
gcn_multimodal_identity test   0.6589  0.9120      0.5538      0.0720
                        train  0.6572  0.9043      0.6184      0.0734
                        val    0.6